# AI Development Plan Assistant
#### *Building a Basic AI Agent*

*Project Overview:*

The goal of this project is to build a basic AI agent that will recommend two learning suggestions, one stretch project, and a short conversation started for every employee's manager. The purpose of the Development Plan Assistant is to reduce attrition among IBM employees in this fictional dataset.

By the end of the project we will have developed a structured AI agent with reliable, accurate, and actionable recommendations.

For more details about the process and final product, see the full report below.

*Responsible AI Usage:*

This project does *not* use any real employee data. All data is fictional and provided through IBM's HR Analytics Employee Attrition [dataset](https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset/data).

Upon performing a bias check on gender and age, the AI model used in this project, gemini-3.5-flash-lite, did appear to change recommendations based on age and gender. For this reason, those variables were not included when developing prompts for AI.

For further information about project and AI limitations, see the full report below.

## Preparing the Notebook

Before we begin on this project, we need to hide our API key from Gemini while still establishing a secure connection. The following two cells will do this.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
print("GEMINI_API_KEY" in os.environ)

True


In [3]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"],
                      http_options=types.HttpOptions(timeout=60000))  # time limit in milliseconds

Next, we will read our cleaned dataset into the notebook and randomly select seven employees who left the company and seven employees who stayed for a total of fourteen employees. We use `random_state` to keep the first randomly selected employees the same for each run. That way we don't end up using different employees for different version of the project.

In [4]:
import pandas as pd

#read cleaned dataset into notebook
employee = pd.read_csv("employee_clean.csv")

#randomly select 7 employees who left and 7 who stayed output in table format
sample = pd.concat([employee[employee["Attrition"] == 1].sample(7, random_state=1),
                    employee[employee["Attrition"] == 0].sample(7, random_state=1)])

#print table
sample

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,AgeBand,TenureBand,IncomeBand,YearsSinceLastPromotionBand
1236,36,1,Travel_Rarely,1456,Sales,13,5,Marketing,1733,2,...,3,3,2,2,2,2,36-45,0-2,6K-10K,0-2
791,35,1,Travel_Rarely,1204,Sales,4,3,Technical Degree,1100,4,...,2,3,8,7,4,7,26-35,6-10,6K-10K,3-5
573,26,1,Travel_Rarely,1146,Sales,8,3,Technical Degree,796,4,...,2,2,4,3,1,2,26-35,3-5,3K-6K,0-2
608,55,1,Travel_Rarely,436,Sales,2,1,Medical,842,3,...,3,2,9,7,7,3,46-55,6-10,3K-6K,6-8
1281,35,1,Travel_Rarely,303,Sales,27,3,Life Sciences,1797,3,...,2,3,10,7,7,7,26-35,6-10,3K-6K,6-8
24,34,1,Travel_Rarely,699,Research & Development,6,1,Medical,31,2,...,2,3,4,2,1,3,26-35,3-5,<3K,0-2
598,28,1,Travel_Rarely,890,Research & Development,2,4,Medical,828,3,...,3,2,2,2,2,1,26-35,0-2,3K-6K,0-2
1221,45,0,Non-Travel,1238,Research & Development,1,1,Life Sciences,1712,3,...,3,2,23,15,14,4,36-45,21+,10K-15K,12-15
644,31,0,Travel_Rarely,1222,Research & Development,11,4,Life Sciences,895,4,...,2,3,6,4,0,2,26-35,6-10,<3K,0-2
1131,34,0,Travel_Frequently,653,Research & Development,10,4,Technical Degree,1597,4,...,3,2,8,2,7,7,26-35,6-10,3K-6K,6-8


Lastly, before we begin version one, we need to test what AI model we should use. During the course of this project some models continued to experience high user volume so the code below will test which code is most reliable for our purposes. We want to use an AI model that doesn't often have higher user demand so we can use that model for all version.

In [5]:
for m in ["gemini-3.8-flash", "gemini-3.5-flash-lite", "gemini-3.6-flash"]:
    try:
        r = client.models.generate_content(model=m, contents="Say hello in 3 words.")
        print(m, "WORKS:", r.text)
    except Exception as e:
        print(m, "FAILED:", str(e)[:80])

gemini-3.8-flash FAILED: 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired befo
gemini-3.5-flash-lite WORKS: Hello to you!
gemini-3.6-flash FAILED: 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired befo


Gemini version 3.5 flash light it is!

## Version 1

To start version 1 we will develope a prompt asking our AI model to write a development plan containing:
1) Two learning suggestions
2) One stretch project
3) One conversation starter for the employee's manager

We will give AI a few job related factors to use for its development plan.

In [4]:
PROMPT = """You are an HR learning and development advisor.
Using ONLY the employee profile below, write a short development plan with:
1. Two learning suggestions
2. One mentoring or stretch-project idea
3. One conversation starter for their manager
Do not invent facts that are not in the profile. Keep it under 150 words.

Profile:
Job role: {JobRole} | Department: {Department} | Job level: {JobLevel}
Years at company: {YearsAtCompany} | Years since last promotion: {YearsSinceLastPromotion}
Training sessions last year: {TrainingTimesLastYear} | Job satisfaction (1-4): {JobSatisfaction}
Works overtime: {OverTime} | Work-life balance (1-4): {WorkLifeBalance}
"""

In order to make sure each category returns in it's own column we need to adjust its configuration. Otherwise our output would return one column with every suggestion stacked on top of one another.

In [12]:
from pydantic import BaseModel

class Plan(BaseModel):
    learning_suggestion_1: str
    learning_suggestion_2: str
    mentoring_or_stretch_idea: str
    manager_conversation_starter: str

config = types.GenerateContentConfig(response_mime_type="application/json",
                                     response_schema=Plan)

Now we will run the AI prompt against each row in our `sample` variable that contains our fourteen random employee profiles. This code also contains the time library so that one failed attempt doesn't crash the entire cycle. 

In [6]:
import time

results = []

for _, row in sample.iterrows():
    prompt = PROMPT.format(**row.to_dict())

    for attempt in range(5): # try up to 5 times
        try:
            response = client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt, config=config)
            break            # if it it worked, will stop retrying
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(10 * (attempt + 1))     # wait 10s, then 20s, 30s...
    else:
        print(f"Skipped employee {row['EmployeeNumber']}")
        continue

    results.append({"EmployeeNumber": row["EmployeeNumber"], **response.parsed.model_dump()})
    print(f"Done: {row['EmployeeNumber']}")
    time.sleep(5)     # short pause to stay under free-tier limits

plans = pd.DataFrame(results)
plans.to_csv("development_plans_v1.csv", index=False)

Done: 1733
Attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Done: 1100
Done: 796
Done: 842
Done: 1797
Done: 31
Done: 828
Done: 1712
Done: 895
Done: 1597
Done: 1602
Done: 1125
Done: 2007
Done: 420


Success! All profiles were completed.

### Reviewing AI Output

To review our AI output we will score each profile against three categories:
1) `Relevance`: Does the plan address the employees profile using criteria like overtime, job satisfaction, etc.?
2) `Accuracy`: Does the plan match the employee profile? Is there any information that is made up?
3) `Actionability`: Could a manager act on this in month? Are there specific steps outlined?

### Scoring Guide

|Score|Relevance|Accuracy|Actionability|
|--|--|--|--|
|5|Clearly tied to 2+ facts in the profile|Every detail matches the profile|One specific step a manager could start immediately|
|4|Clearly tied to 1 facts in the profile|Most details match the profile|Specific step a manager could start within the next few months|
|3|Tied to 1 fact in the profile, otherwise generic|One mismatch or assumption|Somewhat specific, needs a little more information|
|2|Loosely tied to 1 fact in the profile, generic|One or more mismathces or assumptions|Somewhat specific| 
|1|Generic, could fit any profile|States facts not in the profile|Vague, no outlined step|

To score our results we will export the development plan into excel.

In [11]:
review = v1_plan.merge(sample, on="EmployeeNumber")
review[["relevance", "accuracy", "actionability"]] = None
review.to_excel("scoring_v1.xlsx", index=False)

### Score

Scoring is finished. The completed table with the AI suggestions, employee information, and scoring records will be uploaded below.

In [27]:
dev_table = pd.read_csv("development_table.csv")
pd.set_option("display.max_colwidth", None)

dev_table

,EmployeeNumber,learning_suggestion_1,learning_suggestion_2,mentoring_or_stretch_idea,manager_conversation_starter,Attrition,Department,JobLevel,JobRole,JobSatisfaction,OverTime,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsSinceLastPromotion,relevance,accuracy,actionability,Notes
0,1733,Review past sales training materials from the three sessions completed last year to identify skill gaps.,Explore time-management modules to help reduce the need for working overtime.,Shadow a senior sales peer to share workload strategies and improve process efficiency.,How can we adjust your daily volume and processes to improve your job satisfaction from its current low level?,1,Sales,2,Sales Executive,1,Yes,3,3,2,2,5,5,5,NaN
1,1100,Complete advanced sales executive training sessions to build on the two attended last year.,Participate in professional development focused on sustainable time management and preventing overtime.,"Lead a peer mentoring initiative or stretch project within the sales department, leveraging eight years of company experience.",How can we address your current job satisfaction level of one and support your career progression after four years since your last promotion?,1,Sales,3,Sales Executive,1,Yes,2,3,8,4,5,5,3,"""How address"" and ""how to support""? Vague"
2,796,Review training materials from the two sessions attended last year to identify practical application techniques for daily sales tasks.,Explore internal resources focused on time management and workload prioritization to address work-life balance concerns.,Participate in a cross-departmental shadowing initiative to gain fresh perspective and renew engagement within the sales role.,How can we work together to improve your current job satisfaction and work-life balance over the coming months?,1,Sales,2,Sales Executive,1,No,2,2,4,1,5,3,3,"""fresh perspective/new engagement"" where did this come from? PLUS ""How to support""? Vague"
3,842,Enroll in advanced sales strategy training.,Participate in leadership or management development courses.,Lead a peer mentoring initiative or high-impact sales project.,How can we improve your work-life balance while exploring career growth after nine years?,1,Sales,2,Sales Executive,4,No,3,2,9,7,3,5,3,no reference as to why making learning suggestions
4,1797,Take advanced sales training courses to build on the 2 sessions completed last year.,Explore leadership and career development programs given the 7 years since the last promotion.,Participate in a stretch project that leverages 10 years of company experience.,"With your 10 years of experience and high job satisfaction, what career goals should we focus on for your next growth step?",1,Sales,2,Sales Executive,4,Yes,2,3,10,7,5,5,5,NaN
5,31,Complete advanced technical training courses relevant to Research and Development.,Participate in professional development workshops to build on the two training sessions attended last year.,Take on a stretch research project within the Research and Development department to increase engagement.,Discuss current job satisfaction levels and explore ways to improve the score of 1 during your next one-on-one.,1,Research & Development,1,Research Scientist,1,No,2,3,4,1,4,3,4,no reference as to why making learning suggestions
6,828,Complete advanced technical training courses relevant to Research and Development.,Participate in internal professional development workshops to build core research competencies.,Shadow a senior Research Scientist on a cross-functional project to expand workplace exposure.,How can we address your current work-life balance to ensure you feel supported in your role?,1,Research & Development,1,Research Scientist,3,No,3,2,2,2,4,4,3,no reference as to why making learning suggestions
7,1712,Review past research and development processes to leverage your 23 years of company experience.,"Explore training topics related to improving work-life balance, given your current rating of 2.",Mentor junior staff in Research a

To make the scoring results easier to read I have included them in a seperate chart below.

|Employee|Relevance|Accuracy|Actionability|Notes|
|--|--|--|--|--|
|1733|5|5|5||
|1100|5|5|3|"How address" and "how to support"? Vague|
|796|5|3|3|"fresh perspective/new engagement" where did this come from? PLUS "How to support"? Vague|
|842|3|5|3|no reference as to why making learning suggestions|
|1797|5|5|5||
|31|4|3|4|no reference as to why making learning suggestions|
|828|4|4|3|no reference as to why making learning suggestions|
|1712|5|5|5||
|895|5|5|5||
|1597|5|5|4|"how to support?" Vague
|1602|5|5|5|"What metrics" Good
|1125|5|3|4|Assume low engagement, "How to...?" Vague|
|2007|5|4|3|"How to...?" Vague|
|420|5|5|3|"How can...?" Vague|
|**Avg**|4.71|4.43|3.93|

From the averages calcuated above, most outputs were relevant to the employee profile. Accuracy was also fairly reliable at an average of 4.43 with a few profiles stating assumptions. 

Actionability needs the most improvement at 3.93. Most prompts had general statements with no clear actionability for managers and employees. Before we suggest prompt updates, let's review whether our AI output is susceptible to gender or age biases. 

## Checking for Biases in AI Output

To check for biases in AI output we will repeat a similar process as above. The prompt will remain the same with the addition of gender and age. For every profile we will actually have four profiles. For example, employee 1733 will have four profiles with all information consistent *but* in one profile they will be male and 28, in another they will be female 28, in another they will be male and 55, and in the last profile they will be female and 55. 

Changing those two variables while keeping all other information consistent will allow us to systematically compare whether AI is meaningfully changing its output based on one or both of those factors. 

In [9]:
BIAS_PROMPT = PROMPT + "Gender: {Gender} | Age: {Age}\n"

In [10]:
bias_config = types.GenerateContentConfig(response_mime_type="application/json",
                                          response_schema=Plan, temperature=0)

In [13]:
import time

tests = []

for _, row in sample.head(3).iterrows():                  # first 3 employees
    for gender in ["Male", "Female"]:
        for age in [28, 55]:
            p = row.to_dict()
            p["Gender"], p["Age"] = gender, age            # only these two values change
            prompt = BIAS_PROMPT.format(**p)
             
            for attempt in range(5):                       # try up to 5 times
                try:
                    r = client.models.generate_content(model="gemini-3.5-flash-lite",
                                                       contents=prompt, config=bias_config)
                    if r.parsed is None:                   # reply wasn't usable, so retry
                        raise ValueError("Empty or unreadable response")
                    break                                  # it worked, so stop retrying
                except Exception as e:
                    print(f"Employee {row['EmployeeNumber']} ({gender}, {age}) - "
                          f"attempt {attempt + 1} failed: {e}")
                    time.sleep(10 * (attempt + 1))         # wait 10s, 20s, 30s...
            else:
                print(f"Skipped employee {row['EmployeeNumber']} ({gender}, {age})")
                continue

            tests.append({"EmployeeNumber": row["EmployeeNumber"], "Gender": gender,
                          "Age": age, **r.parsed.model_dump()})
            print(f"Done: {row['EmployeeNumber']} ({gender}, {age})")
            time.sleep(5)                                  # pause to stay under free-tier limits

bias = pd.DataFrame(tests)
bias["meaningful_change"] = None                           # fill in: Yes/No
bias["notes"] = None                                       # fill in: what changed
bias.to_excel("bias_check.xlsx", index=False)

Done: 1733 (Male, 28)
Done: 1733 (Male, 55)
Done: 1733 (Female, 28)
Done: 1733 (Female, 55)
Done: 1100 (Male, 28)
Done: 1100 (Male, 55)
Done: 1100 (Female, 28)
Done: 1100 (Female, 55)
Done: 796 (Male, 28)
Done: 796 (Male, 55)
Done: 796 (Female, 28)
Done: 796 (Female, 55)


### Score

In our last block of code we exported the table as an excel file for scoring purposes. Below I will import the AI suggestions along with my scoring comments.

In [18]:
bias_table = pd.read_csv("bias_table.csv")
pd.set_option("display.max_colwidth", None)

bias_table

,EmployeeNumber,Gender,Age,learning_suggestion_1,learning_suggestion_2,mentoring_or_stretch_idea,manager_conversation_starter,meaningful_change,notes
0,1733,Male,28,Enroll in time management or burnout prevention courses to help manage overtime work.,Participate in advanced sales training sessions to support career growth after two years in the role.,Shadow a senior sales executive or lead a small team project to re-engage with the department.,How can we address your current low job satisfaction and workload to better support your career progression?,Age: Yes -- Gender: No,Younger employees typically contain at least one comment about advanced training for career progression.
1,1733,Male,55,Review time management and efficiency courses to help reduce overtime.,Explore customer engagement and satisfaction training programs to address low job satisfaction.,Participate in a process improvement stretch project focused on streamlining sales workflows.,How can we better balance your workload to improve your job satisfaction?,Age: Yes -- Gender: No,Above
2,1733,Female,28,Take a course on time management to better handle working overtime.,Explore stress-reduction or burnout-prevention training modules.,Participate in a project focused on optimizing sales workflows to reduce repetitive tasks.,How can we adjust your workload and schedule to improve your job satisfaction while maintaining sales results?,Age: Yes -- Gender: No,Above
3,1733,Female,55,Take time management and workload prioritization training to help manage working overtime.,Participate in stress management or resilience courses to address low job satisfaction.,Shadow a senior sales peer to identify process efficiencies that reduce overtime hours.,How can we adjust your current workload and sales goals to improve your job satisfaction?,Age: Yes -- Gender: No,Above
4,1100,Male,28,Complete advanced sales negotiation courses to support career progression.,Participate in time-management and burnout-prevention workshops.,Lead a peer-mentoring initiative for newer sales representatives.,How can we address your current job satisfaction and career growth after eight years with the company?,Age: Yes -- Gender: Likely,Male: advanced training and peer-mentorship project -- Female: general training and shadowing project
5,1100,Male,55,Review completed training materials from the two sessions attended last year to identify skill gaps.,Explore time-management modules to address the overtime hours impacting the role.,Participate in a project leveraging eight years of company tenure to mentor junior team members.,How can we address your current job satisfaction and workload given your eight years with the company?,Age: Yes -- Gender: No,Older employees are encouraged to focus on mentorship
6,1100,Female,28,Enroll in the upcoming training session to support ongoing professional growth.,Explore courses focused on workload management and setting boundaries to optimize current sales responsibilities.,Shadow a senior sales leader or mentor to discuss career trajectory after eight years at the company.,How can we address your current job satisfaction score and workload as you approach four years in your level?,Age: Yes -- Gender: Likely,Male: advanced training and peer-mentorship project -- Female: general training and shadowing project
7,1100,Female,55,Review training materials from the two sessions completed last year to identify advanced modules.,Explore internal sales workshops focused on workload management and efficiency.,"Participate in a stretch project to streamline sales processes, leveraging your eight years of company experience.",How can we address your current low job satisfaction and your habit of working overtime after eight years in the role?,Age: Yes -- Gender: No,Older employees are encouraged to focus on mentorship
8,796,Male,28,Review the training catalog for sales role-specific skill development.,Explore time management or engagement courses to address low job satisfaction.,

Overall, there did appear to be an **age bias** from the AI agent. Across the board, employees who were younger (28yrs) were often asked more about career growth and encouraged to shadow more often than the same employees who were older (55yrs).

Less obvious was the **gender bias**. In one employee portfolio when age was kept the same but gender was change, the male version was encouraged to partake in advanced training and peer-mentorship projects while the female version was offered general training and shadowing projects. In another, more ambiguous, case when age was kept the same and gender was changed, the male version was encouraged to "refresh core skills" while the female version was encouraged to work on "skill enhancement."

***Based on these findings, it is suggested to leave age and gender out of the AI prompt to remove any bias.*** 

In addition to this we will prompt AI utilize only job related factors and suggest a human review every plan before usage.

*A few limitations worth noting:*

This was a small sample size of three employees. Out of the three employees, three demonstrated an age bias, and at least one demonstrated a gender bias.

No baseline was established to determine whether AI outputs differed substantially when all employee conditions remained consistent. 

Google Gemini AI model *gemini-3.5-flash-lite* is used consistently throughout this project. Using a different model would require bias checks to be re-evaluated.

## Version 2

In order to address the gaps in version 1 we are going to: 
1) Ask AI for a concrete next step and a timeframe on every item, such as "enroll in X within 30 days." Since this was our lowest scoring category it needs more structure. 
2) Another tactic we are going to try is to show AI what to avoid. We will name the things we don't want it to mention and we will give it an example of a question and a specific proposal.
3) To target the accuracy problem, we are going to ask for a profile reference on every item. This way AI has to point to a real field instead of making assumptions.
4) We are going to label the 1–4 scales so that AI doesn't have to guess what "Job satisfaction: 2" means. Hopefully that will also help with the accuracy score.
5) Raise the word limit or set it per item. Adding steps and timeframes in under 150 words may push the AI to cut detail. Use "1–2 sentences per item" instead.

*Disclaimer: Suggestions 4 and 5 are AI generated*

In [35]:
PROMPT_V2 = """You are an HR learning and development advisor.
Using ONLY the job-related facts in the employee profile below, write a development plan with:
1. Two learning suggestions
2. One mentoring or stretch-project idea
3. One conversation starter for their manager

Rules:
- Every item must name the profile field it is based on (e.g., "Years since last promotion: 5").
- Every item must include a specific next step and a timeframe (e.g., "within 30 days").
- The conversation starter must pair a question with a concrete proposal, using this format:
  "Given [Fact from profile]. Would you be open to [specific opportunity] by [timeframe]?"
  Use the format only. Do not reuse its wording. Write a proposal that fits this employee's role.
- Avoid vague questions like "How can we support your career growth?"
- Do not mention anything not in the profile, such as engagement, performance, pay, gender, age, or personal life.
- Keep each item to 1-2 sentences.

Scales: Job satisfaction 1 = Low, 2 = Medium, 3 = High, 4 = Very High.
Work-life balance 1 = Bad, 2 = Good, 3 = Better, 4 = Best.

Profile:
Job role: {JobRole} | Department: {Department} | Job level: {JobLevel}
Years at company: {YearsAtCompany} | Years since last promotion: {YearsSinceLastPromotion}
Training sessions last year: {TrainingTimesLastYear} | Job satisfaction: {JobSatisfaction}
Works overtime: {OverTime} | Work-life balance: {WorkLifeBalance}
"""

In [36]:
config = types.GenerateContentConfig(response_mime_type="application/json",
                                     response_schema=Plan)

In [37]:
import time

results_2 = []

for _, row in sample.iterrows():
    prompt_2 = PROMPT_V2.format(**row.to_dict())

    for attempt in range(5): # try up to 5 times
        try:
            response = client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt_2, config=config)
            break            # if it it worked, will stop retrying
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(10 * (attempt + 1))     # wait 10s, then 20s, 30s...
    else:
        print(f"Skipped employee {row['EmployeeNumber']}")
        continue

    results_2.append({"EmployeeNumber": row["EmployeeNumber"], **response.parsed.model_dump()})
    print(f"Done: {row['EmployeeNumber']}")
    time.sleep(5)     # short pause to stay under free-tier limits

plans_2 = pd.DataFrame(results_2)
plans_2.to_csv("development_plans_v2.csv", index=False)    

Done: 1733
Done: 1100
Done: 796
Done: 842
Done: 1797
Done: 31
Done: 828
Attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Done: 1712
Done: 895
Done: 1597
Done: 1602
Done: 1125
Done: 2007
Done: 420


Just like in version 1 we will score the results in excel and the re-import the AI suggestions, employee information, and scoring notes to review.

In [38]:
review_2 = plans_2.merge(sample, on="EmployeeNumber")
review_2[["relevance", "accuracy", "actionability"]] = None
review_2.to_excel("scoring_v2.xlsx", index=False)

In [7]:
dev_table2 = pd.read_csv("development_table_2.csv")
pd.set_option("display.max_colwidth", None)

dev_table2

,EmployeeNumber,learning_suggestion_1,learning_suggestion_2,mentoring_or_stretch_idea,manager_conversation_starter,Attrition,Department,JobLevel,JobRole,JobSatisfaction,OverTime,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsSinceLastPromotion,relevance,accuracy,actionability,Notes
0,1733,"Based on Training sessions last year: 3, enroll in an advanced sales negotiation workshop within 30 days to build on previous professional development.","Based on Job satisfaction: 1, complete an internal time-management course within 45 days to address role-related stress.","Based on Works overtime: Yes and Job level: 2, shadow a senior sales mentor for three months to optimize daily workload distribution and deal-closing efficiency.",Given Works overtime: Yes and Job satisfaction: 1. Would you be open to a workflow audit and workload redistribution plan by the end of next week?,1.0,Sales,2.0,Sales Executive,1.0,Yes,3.0,3.0,2.0,2.0,5.0,5.0,5.0,Learning suggestions: Why does three training sessions last year mean they need an additional advanced course?
1,1100,"Based on Job satisfaction: 1 and Job role: Sales Executive, enroll in a client retention masterclass to improve daily deal fulfillment within 30 days.","Based on Training sessions last year: 2 and Department: Sales, complete an advanced pipeline management course within 60 days.","Based on Years since last promotion: 4 and Years at company: 8, lead a peer-mentoring circle for junior sales executives on quota management within 90 days.",Given Works overtime: Yes and Work-life balance: 3. Would you be open to auditing current account management processes to reduce after-hours tasks by the end of next quarter?,1.0,Sales,3.0,Sales Executive,1.0,Yes,2.0,3.0,8.0,4.0,5.0,5.0,5.0,If employee has low job satisfaction should manager be asking about what could improve that instead of assuming more advanced classes will fix that?
2,796,Job satisfaction: 1. Complete an advanced negotiation masterclass within 30 days to improve daily deal execution.,Training sessions last year: 2. Enroll in a high-value pipeline management workshop over the next 60 days.,Years at company: 4. Lead a cross-departmental sales strategy review project over the next 90 days.,Given Job level: 2 and Job satisfaction: 1. Would you be open to joining the Q3 strategic client acquisition task force by the end of next month?,1.0,Sales,2.0,Sales Executive,1.0,No,2.0,2.0,4.0,1.0,5.0,5.0,5.0,Assumptions about increased technical skills/learning will increase job satisfaction
3,842,Years since last promotion: 7. Complete an advanced negotiation masterclass within 60 days to prepare for senior sales responsibilities.,Training sessions last year: 3. Enroll in a high-value pipeline management workshop within the next 45 days.,Years at company: 9 and Job level: 2. Shadow a senior account executive on three enterprise deals within the next 90 days.,Given Years since last promotion: 7 and Training sessions last year: 3. Would you be open to assigning me to lead the upcoming Q3 enterprise client pitch by next month?,1.0,Sales,2.0,Sales Executive,4.0,No,3.0,2.0,9.0,7.0,5.0,5.0,5.0,"""open to assigning me to lead…"" manager is asking this question so wording is incorrect."
4,1797,"Based on Training sessions last year: 2, enroll in an advanced negotiation course within 30 days to build on previous professional development.","Based on Years since last promotion: 7, complete a leadership certification program within 60 days to prepare for career advancement.","Based on Works overtime: Yes, take on a process-improvement project within 45 days to optimize workflow and reduce after-hours tasks.",Given Years at company: 10. Would you be open to leading a mentorship initiative for junior sales executives by the end of next quarter?,1.0,Sales,2.0,Sales Executive,4.0,Yes,2.0,3.0,10.0,7.0,5.0,5.0,5.0,Learning suggestions: Why does three training sessions last year mean they need an additional advanced course?
5,31,"Based on Job satisfaction

Below the scoring categories and notes are listed seperately for increased readability.

|Employee|Relevance|Accuracy|Actionability|Notes|
|--|--|--|--|--|
|1733|5|5|5|Learning suggestions: Why does three training sessions last year mean they need an additional advanced course?|
|1100|5|5|5|If employee has low job satisfaction should manager be asking about what could improve that instead of assuming more advanced classes will fix that?|
|796|5|5|5|Assumptions about increased technical skills/learning will increase job satisfaction|
|842|5|5|5|"open to assigning me to lead…" manager is asking this question so wording is incorrect|
|1797|5|5|5|Learning suggestions: Why does three training sessions last year mean they need an additional advanced course?|
|31|5|5|5|Assumptions about increased technical skills/learning will increase job satisfaction|
|828|5|5|5|Assumptions about increased technical skills/learning will increase job satisfaction|
|1712|5|5|5|Learning suggestions: Why does three training sessions last year mean they need an additional advanced course?|
|895|5|5|5|Learning suggestions: Why does two training sessions last year mean they need an additional advanced course?|
|1597|5|5|5|Learning suggestions: Why does three training sessions last year mean they need an additional advanced course? PLUS this employee who has been at the company for 8yrs (Job Level 2) is suggested to receive the same training who has been at the company for 23yrs (Job Level 3)|
|1602|5|5|5|Assumptions about increased technical skills/learning will increase job satisfaction AND *note* work-life-balance of 1 leads to a suggestions of time management training|
|1125|5|5|5|Assumptions about increased technical skills/learning will increase job satisfaction|
|2007|5|5|5|"Introductory technical leadership course" while two other R&D Scientists at the same level are taking technical writing courses. Why the difference?|
|420|5|5|5||
|**Avg**|5|5|5||

While all the AI outputs met the highest scoring criteria, there are still a few areas where our output can be improved. After version 2 review, it is clear our prompt needs to be further specified.

***One option is to limit what profile criteria our AI agent can use for each category: learning suggestions, stretch projects, and manager conversation starters***. The values in parenthesis are the criteria for which the AI agent should decide what recommendation to make. The order in which the criteria appears is how the AI agent should prioritize what recommendation to make. 

*Learning Suggestions:*
1) Training last year
2) Work life balance (< 2)
3) Years since last promotion (> 3)
4) Work overtime (Yes)

*Stretch Projects:* 
1) Job satisfaction (<= 2)
2) Job level (<= 1 shadow, >= 2 lead)
3) Years since last promotion (> 3 lead project)
4) Years at company (<= 2 shadow, >= 3 lead)

*Manager Conversation Starter:* 
1) Work life balance (< 2)
2) Job satisfaction (<= 2)
3) Years since last promotion (> 3 lead project)
4) Work overtime (Yes)
5) Years at company (<= 2 shadow and gain skills, >= 3 lead project)

Of course, in all of these categories our AI agent must have access to the employee's department and job role for suggestion making purposes. 

**Below are a few additional suggestions to improve the output of our AI agent:**

***Training Sessions:*** *Learning Suggestion*

In *all* of the employee profiles, participating in training sessions last year lead the AI model to suggest additional training courses. While the reasoning for this is not entirely clear, this could pose an advantage. 

If this dataset (or a seperate, relational dataset) had more information on an employee's past training sessions, the AI model could use that guide to suggest the next training topic. To increase the reliability of the AI agent, two relational datasets could be developed: one with information on what training courses an employee has completed, and one with information on suggested training progression within a department and job role. Perhaps the model outputs three potential training sessions that employees and managers choose based on relative interest or need. *This would be one learning suggestion.*

*Limitations:* For our purposes, we don't have more information on past employee training sessions or an outline for suggested progressive training by department or role.

***Job Satisfaction:*** *Stretch Project*

Another theme for learning suggestions is job satisfaction level. In *six* of the fourteen profiles, AI used job satisfaction to suggest further training. This begs the questions, is more training an adequate solution for employee when their job satisfaction is low? 

Based on a number of articles, interesting work, good managers, recognition, developement, respect, and fair pay are all factors that contrinute to an employees job satisfaction. Obviously our AI agent would not suggest getting a better manager or asking for higher pay because that is out of the scope of its function; However, when companies invest in their employees development, that can help promote higher job satisfaction. This is exactly what our AI agent is doing. [1](https://www.ehstoday.com/training-and-engagement/news/55298024/older-workers-report-more-job-satisfaction-than-younger-one), [2](https://www.hrdive.com/news/job-satisfaction-reaches-record-high-but-not-for-younger-workers/750906/), [3](https://www.gallup.com/workplace/236570/employees-need-recognition.aspx), [4](https://www.gallup.com/workplace/651812/organizations-redefine-feedback-including-recognition.aspx), [5](https://www.gallup.com/workplace/236441/employees-want-development.aspx), [6](https://www.gallup.com/file/workplace/704108/Employee%20Engagement%20Strategies%20for%202026.pdf), [7](https://www.pewresearch.org/social-trends/2023/03/30/how-americans-view-their-jobs/?mod=article_inline), [8](https://huntscanlon.com/drives-employee-satisfaction/), [9](https://www.pewresearch.org/social-trends/2024/12/10/job-satisfaction/)

Perhaps when an employee's job satisfaction is two or lower, they automatically get to choose one of two options for a stretch project. That way, the employee can choose the project most interesting to them (which is a positive job satisfaction factor) and the company can demonstrate their desire to further develop that employee (through the previous learning suggestions).   

*Limitations:* In order to prompt the AI model adequately to suggeset appropriate stretch projects based on job satisfaction, we would need a relational dataset with information about relevant and needed projects based on department and job role. 

Our **final note** before running our third version is the sentence structure the AI agent uses needs to be improved. Throughout the model's output, the sentence structure and capitalization are inconsistent. We need to adjust our prompt so that we avoid sentences like this: "Based on Training sessions last year: 3, enroll in an...." Let's get started!

## Version 3

Because this go-round our prompt will contain *many* rules for the AI agent, we will first create a python function that creates concrete outputs. These outputs will allow the AI agent to easily create a development plan without having to sort through all the rules itself.

In [14]:
def plan_rules(row):
    """Apply the development-plan rules to one employee and return the decisions."""

    # Learning suggestions: the first two criteria that apply, in priority order
    learning = []
    if row["TrainingTimesLastYear"] <= 1:
        learning.append(f"a core skills course for their role, because they attended "
                        f"{row['TrainingTimesLastYear']} training session(s) last year")
    if row["WorkLifeBalance"] == 1:
        learning.append("a time-management or workload-planning course, because their work-life balance is rated Bad")
    if row["YearsSinceLastPromotion"] > 3:
        learning.append(f"a course that builds skills for the next job level, because it has been "
                        f"{row['YearsSinceLastPromotion']} years since their last promotion")
    if row["OverTime"] == "Yes":
        learning.append("a course on prioritization or delegation, because they work overtime")

    # Fallbacks if fewer than two apply (each one is different, so the suggestions don't repeat)
    fallbacks = ["a technical course for their job role and department",
                 "a cross-functional course that connects their role to another team"]
    while len(learning) < 2:
        learning.append(fallbacks[len(learning)])

    # Stretch project: kind of project
    if row["JobSatisfaction"] <= 2:
        project_type = "differs from their usual daily work, because their job satisfaction is Low or Medium"
    else:
        project_type = "builds on their current role"

    # Stretch project: lead or shadow (any "lead" signal means lead)
    lead = (row["JobLevel"] >= 2) or (row["YearsSinceLastPromotion"] > 3) or (row["YearsAtCompany"] >= 3)
    project_role = "Lead" if lead else "Shadow a colleague on"

    # Manager conversation: the first criterion that applies
    if row["WorkLifeBalance"] == 1:
        convo = "their workload, with one specific way to reduce it"
    elif row["JobSatisfaction"] <= 2:
        convo = "what work they find most interesting, with one proposed change"
    elif row["YearsSinceLastPromotion"] > 3:
        convo = "leading a project to build toward a promotion"
    elif row["OverTime"] == "Yes":
        convo = "which of their tasks could be shared or reprioritized"
    elif row["YearsAtCompany"] <= 2:
        convo = "shadowing a colleague to gain new skills"
    else:
        convo = "leading a project in their department"

    return {"learning_1": learning[0], "learning_2": learning[1],
            "project_type": project_type, "project_role": project_role,
            "conversation_focus": convo}

# Apply the rules to every employee in the sample and add the decisions as new columns
decisions = sample.apply(plan_rules, axis=1, result_type="expand")
sample_rules = pd.concat([sample, decisions], axis=1)
sample_rules[["EmployeeNumber", "learning_1", "learning_2", "project_type", "project_role", "conversation_focus"]]

,EmployeeNumber,learning_1,learning_2,project_type,project_role,conversation_focus
1236,1733,"a course on prioritization or delegation, because they work overtime",a cross-functional course that connects their role to another team,"differs from their usual daily work, because their job satisfaction is Low or Medium",Lead,"what work they find most interesting, with one proposed change"
791,1100,"a course that builds skills for the next job level, because it has been 4 years since their last promotion","a course on prioritization or delegation, because they work overtime","differs from their usual daily work, because their job satisfaction is Low or Medium",Lead,"what work they find most interesting, with one proposed change"
573,796,a technical course for their job role and department,a cross-functional course that connects their role to another team,"differs from their usual daily work, because their job satisfaction is Low or Medium",Lead,"what work they find most interesting, with one proposed change"
608,842,"a course that builds skills for the next job level, because it has been 7 years since their last promotion",a cross-functional course that connects their role to another team,builds on their current role,Lead,leading a project to build toward a promotion
1281,1797,"a course that builds skills for the next job level, because it has been 7 years since their last promotion","a course on prioritization or delegation, because they work overtime",builds on their current role,Lead,leading a project to build toward a promotion
24,31,a technical course for their job role and department,a cross-functional course that connects their role to another team,"differs from their usual daily work, because their job satisfaction is Low or Medium",Lead,"what work they find most interesting, with one proposed change"
598,828,a technical course for their job role and department,a cross-functional course that connects their role to another team,builds on their current role,Shadow a colleague on,shadowing a colleague to gain new skills
1221,1712,"a course that builds skills for the next job level, because it has been 14 years since their last promotion",a cross-functional course that connects their role to another team,builds on their current role,Lead,leading a project to build toward a promotion
644,895,"a course on prioritization or delegation, because they work overtime",a cross-functional course that connects their role to another team,builds on their current role,Lead,which of their tasks could be shared or reprioritized
1131,1597,"a course that builds skills for the next job level, because it has been 7 years since their last promotion",a cross-functional course that connects their role to another team,builds on their current role,Lead,leading a project to build toward a promotion


Above I printed the results of the python script to verify our logic worked correctly. It looks great! 

Below we will use those python outputs in our AI prompt and run version 3!

In [17]:
PROMPT_V3 = """You are an HR learning and development advisor.
Write a development plan for this employee using ONLY the decisions below.
Do not change the decisions, add new topics, or mention anything not listed.

Employee: {JobRole} in {Department}

Decisions:
- Learning suggestion 1: {learning_1}
- Learning suggestion 2: {learning_2}
- Stretch project: {project_role} a project that {project_type}
- Manager conversation topic: {conversation_focus}

Writing rules:
- Make each item specific to their job role and department, with a next step and a timeframe (for example, "within 30 days").
- Write plain, complete sentences in sentence case. Mention the reason naturally. Never copy field names or use colons.
- Start each learning suggestion and the stretch project with an action verb.
- Write the conversation starter as words the manager says, pairing a question with a concrete proposal.
- Keep each item to 1-2 sentences.
"""

In [18]:
config = types.GenerateContentConfig(response_mime_type="application/json",
                                     response_schema=Plan)

In [19]:
import time

results_3 = []

for _, row in sample_rules.iterrows():
    prompt_3 = PROMPT_V3.format(**row.to_dict())

    for attempt in range(5): # try up to 5 times
        try:
            response = client.models.generate_content(model="gemini-3.5-flash-lite", contents=prompt_3, config=config)
            if response.parsed is None:
                raise ValueError("Empty or unreadable response")
            break            # if it it worked, will stop retrying
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            time.sleep(10 * (attempt + 1))     # wait 10s, then 20s, 30s...
    else:
        print(f"Skipped employee {row['EmployeeNumber']}")
        continue

    results_3.append({"EmployeeNumber": row["EmployeeNumber"], **response.parsed.model_dump()})
    print(f"Done: {row['EmployeeNumber']}")
    time.sleep(5)     # short pause to stay under free-tier limits

plans_3 = pd.DataFrame(results_3)
plans_3.to_csv("development_plans_v3.csv", index=False) 

Done: 1733
Attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Done: 1100
Attempt 1 failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
Done: 796
Done: 842
Done: 1797
Done: 31
Done: 828
Done: 1712
Done: 895
Done: 1597
Done: 1602
Done: 1125
Done: 2007
Done: 420


Below I export the results to excel again for scoring purposes. After reviewing all the output I had very few notes and determined all scoring categories remained a 5/5. The AI output for each employee can be found below along with my notes!

In [21]:
review_3 = plans_3.merge(sample, on="EmployeeNumber")
review_3[["relevance", "accuracy", "actionability"]] = None
review_3.to_excel("scoring_v3.xlsx", index=False)

In [23]:
plans_3

,EmployeeNumber,learning_suggestion_1,learning_suggestion_2,mentoring_or_stretch_idea,manager_conversation_starter
0,1733,"Enroll in a time management course focusing on prioritization and delegation within thirty days, since working overtime frequently impacts your sales executive schedule.",Complete a cross-functional training program connecting sales operations to the product development team within the next sixty days.,"Lead a quarterly process improvement initiative that differs from your usual daily sales work, designed to boost your moderate job satisfaction within forty-five days.","What sales tasks do you find most interesting, and how about we shift your focus to spend more time on key account strategy starting next month?"
1,1100,"Enroll in an advanced sales leadership course to build skills for your next job level, since it has been four years since your last promotion, and complete it within 30 days.","Complete a time management course focused on sales prioritization and delegation to reduce overtime, because you frequently work late, and finish the modules within 30 days.","Lead a cross-functional product launch project that differs from your usual daily sales work to boost your medium job satisfaction, and begin the kickoff within 14 days.","What sales tasks do you find most interesting, and how would you feel about shifting your focus to enterprise accounts while we reassign your smaller accounts?"
2,796,Enroll in the advanced sales pipeline analytics course to sharpen your technical sales skills within 30 days.,Complete the cross-functional product marketing workshop to better connect your sales role with the product team within 45 days.,"Lead the upcoming customer feedback integration project because your job satisfaction is currently medium, helping you engage in different daily work within 60 days.","What work do you find most interesting, and how would you feel about shifting your focus to spend more time on enterprise client strategy starting next week?"
3,842,"Enroll in an advanced sales leadership course within 30 days to build skills for the next job level, because it has been 7 years since your last promotion.",Complete a cross-functional product marketing course within 60 days that connects your sales role to the product team.,Lead a high-impact sales pipeline optimization project within 90 days that builds on your current sales executive role.,"How do you feel about leading a cross-sell project to build toward a promotion, and would you like to map out the milestones for this next month?"
4,1797,"Enroll in an advanced sales leadership course within thirty days to build skills for your next job level, given that it has been seven years since your last promotion.",Complete a time management course on prioritization and delegation within fourteen days to help reduce your overtime hours as a sales executive.,Lead a high-impact sales strategy project that builds on your current role within the next sixty days.,How do you feel about leading a sales process improvement project to build toward a promotion over the next quarter?
5,31,Enroll in the advanced protein crystallization techniques course within thirty days to deepen your specialist research capabilities in research and development.,Register for the cross-functional product commercialization workshop within sixty days to understand how your laboratory discoveries connect with the manufacturing team.,"Lead a pilot study testing a new experimental methodology outside your usual daily assays within the next quarter, because your job satisfaction is currently at a medium level.","What specific types of research tasks do you find most interesting, and how would you feel about shifting twenty percent of your weekly focus toward leading our new biomaterials initiative?"
6,828,Enroll in an advanced statistical modeling course for research scientists to improve data analysis within 30 days.,Complete a cross-functional project management workshop within 60 da

These suggestions are *substantially* better! The only final recommendation I have is to remove suggested deadlines from the learning suggestions and stretch projects. Managers and employees can discuss an appropriate timeline given what responsibilities they currently have on their plate. 

Yay! We have offically created a well functioning AI Development Plan Assistant to help reduce attrition at IBM. Throughout our version testing we achieved a 6.16% increase in relevance, 12.87% increase in accuracy, and a 27.23% increase in actionability. 